# Language Modeling

Estimating the joint probability $p(\mathbf{x}_1, \ldots,\mathbf{x}_T)$ of a sequence of discrete tokens prove useful for various reasons. This task is called *language modeling*.
For instance, machine translation or ASR systems generate sequences by optimizing for the most probable ones. In particular, models which predicts the next element of a sequence are referred to as a **language model** (LM). Recall that we can write a joint distribution as a chain of conditional distributions:

$$
p(\mathbf{x}_1, \ldots,\mathbf{x}_T) = p(\mathbf{x}_1) \prod_{t = 2}^{T} p(\mathbf{x}_{t} \mid \mathbf{x}_{1}, \ldots, \mathbf{x}_{t-1}).
$$

Hence, the output of a model for discrete data must be a distribution $p(\mathbf{x}_{t} \mid \mathbf{x}_{1}, \ldots, \mathbf{x}_{t-1})$ for each token instead of expected values for regression models. In practice, this means that we need to have a finite collection of valid tokens called a **vocabulary**. Then, we can generate text, simply by drawing one token at a time $\mathbf{x}_t \sim p(\mathbf{x}_t \mid \mathbf{x}_1, \ldots, \mathbf{x}_{t-1})$. For example,

$$
\begin{aligned}
&\;p(\text{deep}, \text{learning}, \text{is}, \text{fun}) \\
=& \;p(\text{deep}) \cdot p(\text{learning} \mid \text{deep}) \cdot p(\text{is} \mid \text{deep}, \text{learning}) \cdot p(\text{fun} \mid \text{deep}, \text{learning}, \text{is}).
\end{aligned}
$$

The probabilities can be estimated using [relative frequencies](https://en.wikipedia.org/wiki/Empirical_probability) perhaps with [Laplace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing):

$$
p(\text{deep} \mid \text{learning}) \approx \frac{\#(\text{deep},\, \text{learning}) + \kappa}{\#(\text{learning}) + \kappa|\mathcal{V}|}
$$ 

where $\kappa > 0$ can be thought of as *pseudo-count*. Observe that the smoothing parameter $\kappa$ acts as a regularizer when $\kappa \gg 1,$ where the distribution becomes uniform. Moreover, we usually truncate the context to a fixed number of terms as a Markov hypothesis, and because *n*-grams become sparse in naturally occuring text as *n* increases.

<br>

## Perplexity

Next, we need a generic metric to measure the quality of the language model.
One way is to check how *surprising* the text is.
A good language model is able to predict, with high accuracy, the tokens that come next.
Consider the following continuations of the phrase "It is sunny", as proposed by three different language models:

```text
1. It is sunny outside
2. It is sunny banana tree
3. It is sunny soiupt;mkj ldfosim
```

The first example is clearly the best, although not necessarily factual or accurate, model predicts kind of word correctly. The next is nonsensical, but at least model has learned some degree of correlation between words ('banana' and 'tree'). Finally, the last example 
indicates poor training.

To evaluate a language model, we can use the cross-entropy on the next token which is equivalent to maximizing the likelihood of a text. We normalize this over the number of tokens predicted. For example, we evaluate the model on contexts of variable length $\delta = 1, \ldots, T$ starting from $\mathbf{x}_{t}$:

$$
\mathcal{L} = -\frac{1}{n}\sum_{t}\sum_{\delta = 1}^{T} \log p(\mathbf{x}_{t + \delta} \mid \mathbf{x}_{t}, \ldots, \mathbf{x}_{t + \delta - 1})
$$

where $n$ is the number predictions. For a classifier that predicts all tokens uniformly random, then $\mathcal{L} = \log |\mathcal{V}|$ where $\mathcal{V}$ is the set of tokens. This is a useful baseline. A similarly simple model predicts prior probabilities based on counts of each token in the training data.

In [ ]:
import math
import torch
import torch.nn.functional as F

# Reduction over B × T elements
B, C, T = 32, 28, 128
print(F.cross_entropy(torch.rand(B, C, T), target=torch.randint(C, size=(B, T))))
print(math.log(C))

Historically, researchers in NLP have also used *perplexity* (PP) which is simply the exponential of the cross-entropy:

$$
\text{PP} = \exp\left(-\frac{1}{n}\sum_{t}\sum_{\delta = 1}^{T} \log p(\mathbf{x}_{t + \delta} \mid \mathbf{x}_{t}, \ldots, \mathbf{x}_{t + \delta - 1})\right).
$$

Note that perplexity is equivalent to an inverse likelihood, and to the geometric mean of $\frac{1}{p(\mathbf{x}_t \mid \mathbf{x}_{<t})}$: 

$$
\text{PP} = \frac{1}{\sqrt[n]{\prod_{t}\prod_{\delta = 1}^{T} p(\mathbf{x}_{t + \delta} \mid \mathbf{x}_{[t:\,t + \delta-1]})}} = \sqrt[n]{\prod_{t}\prod_{\delta = 1}^{T} \frac{1} {p(\mathbf{x}_{t + \delta} \mid \mathbf{x}_{[t:\,t + \delta-1]})}}.
$$

Hence, for a perfect model, $\text{PP} = 1.$ On the other hand, if the model predicts $p \approx 0$ for the correct token at one step, then[^pp-bound] we get $\text{PP} = \infty.$ As a baseline, for a uniformly random model, we have $\text{PP} = |\mathcal{V}|.$ This provides a nontrivial upper bound that any useful model must beat. So, we have $\text{PP}$ values $\infty > |\mathcal{V}| \geq 1$ for the three regimes[^pp-regimes]. This can be interpreted as the average number of tries to get the correct prediction at each step, e.g. single try for a perfect model, or $|\mathcal{V}|$ tries for a uniformly random model.

**Remark.** For the sake of concreteness, we evaluated cross-entropy over predictions with context of varying length $\delta = 1, \ldots, T$ from $t.$ But we can also use fixed-length contexts, depending on the given task. In general, we simply evaluate cross-entropy over all instances of next-token prediction regardless of the particulars of the prediction process.

[^pp-bound]: More precisely, for any $\epsilon > 0$, if $p_{t + \delta} \leq \epsilon$ for some $(t, \delta)$, then $\text{PP} \geq \epsilon^{-1/n}.$ 

[^pp-regimes]: The regimes correspond to $\infty > \log |\mathcal{V}| \geq 0$ with cross-entropy.

# Getting sequence data

In this section, we outline the process of extracting sequence data from text. In particular, we will extract character sequences from H. G. Wells' [*The Time Machine*](http://www.gutenberg.org/ebooks/35), a book containing just over 30,000 words. While real applications will typically involve significantly larger datasets, this is sufficient to demonstrate the preprocessing pipeline.

## Processing the sample text

In [ ]:
!mkdir ./data
!curl "https://www.gutenberg.org/cache/epub/35/pg35.txt" --output ./data/time_machine.txt

This has some boilerplate text by [Project Gutenberg](https://www.gutenberg.org/)[^gutenberg] that we have to remove:

[^gutenberg]: Project Gutenberg is an excellent source of literary text data. Moreover, the preprocessing steps for `.txt` files are very similar for each text.

In [ ]:
text = open("./data/time_machine.txt").read()
print(text[:891])

Simply find the start and end of the Project Gutenberg markers:

In [ ]:
start = "*** START OF THE PROJECT GUTENBERG EBOOK THE TIME MACHINE ***"
end = "*** END OF THE PROJECT GUTENBERG EBOOK THE TIME MACHINE ***"
text = text[text.find(start) + len(start): text.find(end)]
print(len(text.split()))
print(text[:129].strip())
print("...\n")
print(text[-95:].strip())

<br>

## Tokenization

Tokens are the *atomic* units of text. Each time step corresponds to one token, but what it is that constitutes a token is a design choice. For example, we can represent the sentence "Deep learning is fun" as a sequence of 4 tokens, with one token for every English word. Then, the set of all words comprise a large vocabulary (typically ~10-100K words). 
Or we can represent the same sentence as a much longer sequence of 30 characters, using a much smaller vocabulary (256 ASCII characters). There is some tradeoff associated with the choice of **vocabulary**[^vocab-tradeoff]. 

Our implementation of the **tokenizer** builds[^standard-vocab] the vocabulary from the text which is represented as a single large string. Tokenization implemented as `tokenize` refers to converting a string to a list of tokens. The vocabulary refers to the list of all tokens. Finally, the tokenizer implements an `encode` method which converts a string to a list of integer indices, and a `decode` method which converts a list of integers to a string.

[^vocab-tradeoff]: For example, using a larger vocabulary provides richer, context-aware understanding compared to ASCII-based tokenization. ASCII-only tokenization, such as treating each character as a token, loses essential semantic and syntactic information. Tokenization has [profound implications](https://x.com/karpathy/status/1759996551378940395) which we will cover in a future chapter.

[^standard-vocab]: In practice, [standard vocabularies](https://huggingface.co/docs/transformers/en/tokenizer_summary) already exist and are re-used for various use cases.

In [ ]:
import re
from collections import Counter
from typing import Union, Optional, TypeVar, List

T = TypeVar("T")
ScalarOrList = Union[T, List[T]]


class Vocab:
    def __init__(self, 
        text: str, 
        min_freq: int = 0, 
        reserved_tokens: Optional[List[str]] = None,
        preprocess: bool = True
    ):
        text = self.preprocess(text) if preprocess else text
        tokens = list(text)
        counter = Counter(tokens)
        reserved_tokens = reserved_tokens or []
        self.token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.itos = [self.unk_token] + reserved_tokens + [tok for tok, f in filter(lambda tokf: tokf[1] >= min_freq, self.token_freqs)]
        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)
    
    def __getitem__(self, tokens: ScalarOrList[str]) -> ScalarOrList[int]:
        if isinstance(tokens, str):
            return self.stoi.get(tokens, self.unk)
        else:
            return [self.__getitem__(tok) for tok in tokens]

    def to_tokens(self, indices: ScalarOrList[int]) -> ScalarOrList[str]:
        if isinstance(indices, int):
            return self.itos[indices]
        else:
            return [self.itos[int(index)] for index in indices]
            
    def preprocess(self, text: str):
        return re.sub("[^A-Za-z]+", " ", text).lower().strip()

    @property
    def unk_token(self) -> str:
        return "▮"

    @property
    def unk(self) -> int:
        return self.stoi[self.unk_token]

    @property
    def tokens(self) -> List[int]:
        return self.itos

For simplicity (i.e. to get smaller models), the source text is preprocessed by removing punctuation and ignoring capitalization. This results in a significantly smaller vocabulary, trading off punctuation and capitalization which are important for text understanding and generating nuanced text.

In [ ]:
import torch

class Tokenizer:
    def __init__(self, vocab: Vocab):
        self.vocab = vocab

    def tokenize(self, text: str) -> List[str]:
        UNK = self.vocab.unk_token
        tokens = self.vocab.stoi.keys()
        return [c if c in tokens else UNK for c in list(text)]

    def encode(self, text: str) -> torch.Tensor:
        x = self.vocab[self.tokenize(text)]
        return torch.tensor(x, dtype=torch.int64)

    def decode(self, indices: Union[ScalarOrList[int], torch.Tensor]) -> str:
        return "".join(self.vocab.to_tokens(indices))

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

Since our vocab includes only lowercase letters, 
the unknown token ▮ replaces missing characters when we encode the text:

In [ ]:
text = open("./data/time_machine.txt").read()
vocab = Vocab(text)
tokenizer = Tokenizer(vocab)
print("vocab:", ", ".join(vocab.itos[:10]) + ", ...")
print("tokenization:", tokenizer.tokenize("Hello!"))
print("encoding-decoding:", "\nHello! ->", tokenizer.encode("Hello!"), "->", tokenizer.decode(tokenizer.encode("Hello!")))

Defining the class for processing the dataset:

In [ ]:
import re
import os
import requests

from pathlib import Path

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)


class TimeMachine:
    def __init__(self, download=False, path=None):
        DEFAULT_PATH = str((DATA_DIR / "time_machine.txt").absolute())
        self.filepath = path or DEFAULT_PATH
        if download or not os.path.exists(self.filepath):
            self._download()
        
    def _download(self):
        url = "https://www.gutenberg.org/cache/epub/35/pg35.txt"
        print(f"Downloading text from {url} ...", end=" ")
        response = requests.get(url, stream=True)
        response.raise_for_status()
        print("OK!")
        with open(self.filepath, "wb") as output:
            output.write(response.content)
        
    def _load_text(self):
        with open(self.filepath, "r") as f:
            text = f.read()
        s = "*** START OF THE PROJECT GUTENBERG EBOOK THE TIME MACHINE ***"
        e = "*** END OF THE PROJECT GUTENBERG EBOOK THE TIME MACHINE ***"
        return text[text.find(s) + len(s): text.find(e)]
    
    def build(self, vocab: Optional[Vocab] = None):
        self.text = self._load_text()
        vocab = vocab or Vocab(self.text)
        tokenizer = Tokenizer(vocab)
        encoded_text = tokenizer.encode(vocab.preprocess(self.text))
        return encoded_text, tokenizer

Basic usage starts with download and then build:

In [ ]:
tm = TimeMachine(download=True)
encoded_text, tokenizer = tm.build()

The `encoded_text` is the encoded clean text. This will be used later as training data.

In [ ]:
print(len(encoded_text), tokenizer.vocab_size)
print(tokenizer.decode(encoded_text[:100]) + "...")

<br>

## Appendix: Zipf's law

[Zipf's law](https://en.wikipedia.org/wiki/Zipf%27s_law) is the observation that token frequency follows an inverse power law. More precisely, it states that 
the frequency $f_i$ of the $i$-th most frequent word for texts in natural language decays inversely proportional to its word rank $i,$ after a few exceptions. Let $a > 0$ characterize the rate of token frequency decay. Then,

$$f_i = {f_1} \cdot {i^{-a}}$$

or

$$\log f_i = -a \log i + \log f_1.$$

Note that the indexing drops a few words, i.e. $i = 1$ corresponds to the rank of the first word that wasn't dropped. The parameter $a$ and the number of skipped words is particular to the text and, to a larger scale, the language used.

In [ ]:
tm = TimeMachine(download=False)
data, tokenizer = tm.build()

freqs = lambda l: list(map(lambda z: z[1], sorted(Counter(l).items(), key=lambda x: x[1], reverse=True)))
words = tokenizer.vocab.preprocess(tm.text).split()
word_freqs = freqs(words)
bigram_freqs = freqs(["__".join(pair) for pair in zip(words[:-1], words[1:])])
trigram_freqs = freqs(["__".join(triple) for triple in zip(words[:-2], words[1:-1], words[2:])])

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "svg"
import matplotlib.pyplot as plt

plt.plot(word_freqs, label="words")
plt.plot(bigram_freqs, label="bigram")
plt.plot(trigram_freqs, label="trigram")
plt.yscale("log")
plt.xscale("log")
plt.ylabel("frequency $f_i$")
plt.xlabel("rank $i$")
plt.legend();

Estimating $a$:

In [ ]:
import math

print("1:", (math.log(word_freqs[100]) - math.log(word_freqs[10])) / (math.log(100) - math.log(10)))
print("2:", (math.log(bigram_freqs[100]) - math.log(bigram_freqs[10])) / (math.log(100) - math.log(10)))
print("3:", (math.log(trigram_freqs[100]) - math.log(trigram_freqs[10])) / (math.log(100) - math.log(10)))

**Remark.** Similar behavior and similar $a$'s have been observed for other novels. For example, *The Time Machine* and *Frankenstein* have similar values. Other types such as non-fiction or plays (*Romeo and Juliet*) have slightly but significantly different values.

# Counting *n*-grams

We develop a simple language model based on *n*-grams.

In [ ]:
import math
import torch
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib_inline import backend_inline

FRAC_LIMIT = 0.3
PAD_TOKEN = "."

DATASET_DIR = Path("./data").absolute()
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

warnings.simplefilter(action="ignore")
backend_inline.set_matplotlib_formats("svg")

Getting the dataset:

In [ ]:
import os
if not os.path.isfile("./data/surnames_freq_ge_100.csv"):
    !wget -O ./data/surnames_freq_ge_100.csv https://raw.githubusercontent.com/particle1331/spanish-names-surnames/master/surnames_freq_ge_100.csv
    !wget -O ./data/surnames_freq_ge_20_le_99.csv https://raw.githubusercontent.com/particle1331/spanish-names-surnames/master/surnames_freq_ge_20_le_99.csv
else:
    print("Data files already exist.")

Loading and preprocessing into a list of strings:

In [ ]:
def load_surnames(frac: float = FRAC_LIMIT, min_len=2) -> list[str]:
    """Load shuffled surnames from files into a list."""

    col = ["surname", "frequency_first", "frequency_second", "frequency_both"]
    filepaths = ["surnames_freq_ge_100.csv", "surnames_freq_ge_20_le_99.csv"]
    dfs = [pd.read_csv(DATASET_DIR / f, names=col, header=0) for f in filepaths]
    df = pd.concat(dfs, axis=0)[["surname"]].sample(frac=frac)
    df = df.reset_index(drop=True)
    df["surname"] = df["surname"].map(lambda s: s.lower())
    df["surname"] = df["surname"].map(lambda s: s.replace("de la", "dela"))
    df["surname"] = df["surname"].map(lambda s: s.replace(" ", "_"))
    df = df[["surname"]].dropna().astype(str)

    names = [
        n for n in df.surname.tolist() 
        if ("'" not in n) and ('ç' not in n) and (len(n) >= min_len)
    ]
    
    return names

In [ ]:
names = load_surnames()
for j in range(5):
    print(names[j])

We will tokenize the names at the character level:

In [ ]:
from collections import Counter

name_lengths = Counter([len(n) for n in names])
corpus = "".join(names)
print("num chars:  ", len(set(corpus)))
print("total names:", len(names))
print("total chars:", len(corpus))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
vocab = Vocab(corpus, preprocess=False)
ax[0].bar(x=[tok for tok, _ in vocab.token_freqs], height=[freq for _, freq in vocab.token_freqs])
ax[0].set_xlabel("character", fontsize=10)
ax[0].set_ylabel("count", fontsize=10)

ax[1].set_xlabel("name length")
ax[1].set_ylabel("count")
ax[1].bar(name_lengths.keys(), name_lengths.values(), color="C1")

fig.tight_layout();

Spanish surnames have median name length 7 (counting space):

In [ ]:
print("range: ", [min(name_lengths.keys()), max(name_lengths.keys())])
print("median: ", int(np.median(sorted([len(n) for n in names]))))

Defining the dataset of context-target pairs. These are constructed by iterating with a window of fixed size over each name to get
$(\mathbf{x}_{i}, \ldots, \mathbf{x}_{i + c - 1}) \mapsto \mathbf{x}_c$ pairs where $i$ is the index and $c$ is the block size. Moreover, each name is padded with padding `.` indicating the start and end of a name, e.g. `"...durana."` for context size 3. Note that all inputs have the same size.

In [ ]:
import torch
from typing import List
from torch.utils.data import Dataset

class CharDataset(Dataset):
    def __init__(self, 
        names: List[str], 
        block_size: int,
        vocab: Optional[Vocab] = None
    ):
        self.block_size = block_size
        self.vocab = vocab or Vocab(text="".join(names), preprocess=False, reserved_tokens=[PAD_TOKEN])
        self.tokenizer = Tokenizer(self.vocab)
        self.xs, self.ys = self.samples(names)

    def vocab_size(self):
        return len(self.vocab)
    
    def __len__(self):
        return len(self.xs)
    
    def __getitem__(self, i: int):
        x = self.tokenizer.encode(self.xs[i])
        y = self.tokenizer.encode(self.ys[i])[0]
        return x, y

    def samples(self, names: List[str]):
        xs, ys = [], []
        for name in names:
            context = PAD_TOKEN * self.block_size
            for c in name + PAD_TOKEN:
                xs.append(context)
                ys.append(c)
                context = context[1:] + c
        return xs, ys

Using this we can create sequence datasets of any **fixed** context size to predict the next character:

In [ ]:
dataset = CharDataset(names, block_size=3)
x, y = zip(*[dataset[i] for i in range(7)])
pd.DataFrame({"xs": dataset.xs[:7], "ys": dataset.ys[:7], "x": list(map(lambda e: e.tolist(), x)), "y": list(map(lambda e: e.item(), y))})

<br>

## Frequency table

Bigrams are essentially input-output pairs with context size 1. The following constructs the table of character bigrams appearing in names in the training dataset. An entry `[i, j]` in the array below is the count of bigrams that start with the `i`th character followed by the `j`th character. We also consider padding.

In [ ]:
split_point = int(0.80 * len(names))
names_train = names[:split_point]
names_valid = names[split_point:]
vocab = Vocab(text="".join(names), preprocess=False, reserved_tokens=[PAD_TOKEN])
bigram_train = CharDataset(names_train, block_size=1, vocab=vocab)
bigram_valid = CharDataset(names_valid, block_size=1, vocab=vocab)

# count matrix
n = bigram_train.vocab_size()
N2 = torch.zeros((n, n), dtype=torch.int32)
for x, y in bigram_train:
    N2[x[0].item(), y.item()] += 1

**Remark.** Note that we use one vocabulary for training and validation (i.e. combined). 

In [ ]:
# viz (ignore <unk>)
plt.figure(figsize=(18, 18))
plt.imshow(N2[1:, 1:], cmap='Blues')
for i in range(N2.shape[0] - 1):
    for j in range(N2.shape[1] - 1):
        chstr = bigram_train.tokenizer.decode(i + 1) + bigram_train.tokenizer.decode(j + 1)
        plt.text(j, i, chstr, ha="center", va="bottom", color="gray")
        plt.text(j, i, N2[1:, 1:][i, j].item(), ha="center", va="top", color="gray")

plt.axis("off");

**Remark.** The count table for general *n*-grams is a $| \mathcal{V} |^n$ matrix. But by Zipf's law, this matrix becomes increasingly sparse with increasing *n*.

<br>

## Modeling counts

The above frequency table can be constructed with $n$-grams, in general. A context $\mathbf{x}$ represented as an $n - 1$ tuple can be used to index the frequency table to get the frequency distribution for the target token. The model outputs the distribution $f(\mathbf{x}) = p(\cdot \mid \mathbf{x})$ by normalizing and smoothing the counts.

In [ ]:
class CountingModel:
    def __init__(self, block_size: int, vocab_size: int, alpha=0.01):
        """Model of observed n-grams to estimate next char probability."""
        self.P = None                    # cond. prob
        self.N = None                    # counts
        self.alpha = alpha               # laplace smoothing
        self.block_size = block_size
        self.vocab_size = vocab_size

    def __call__(self, x: torch.tensor) -> torch.tensor:
        # tuple(x.T) = ([x11, x21, x31], [x12, x22, x32]) 
        # i.e. len = block_size, num entries = B
        # then, P[tuple(x.T)][b] == P[xb1, xb2], so output has shape (B, vocab_size) 
        return torch.tensor(self.P[tuple(x.T)])    

    def fit(self, dataset: CharDataset):
        v = self.vocab_size
        n = self.block_size + 1     # +1 for output dim
        self.N = torch.zeros([v] * n, dtype=torch.int32)  
        for x, y in dataset:
            self.N[tuple(x)][y] += 1

        a = self.alpha
        self.P = (self.N + a)/ (self.N + a).sum(dim=-1, keepdim=True)

    def evaluate(self, dataset: CharDataset):
        loss = 0.0
        for x, y in dataset:
            loss += -torch.log(self(x[None, :])[0, y]).item()
        return loss / len(dataset)

Fitting a bigram model which has a context of one character: 

In [ ]:
vocab = Vocab(text="".join(names), preprocess=False, reserved_tokens=[PAD_TOKEN])
bigram_train = CharDataset(names_train, block_size=1, vocab=vocab)
bigram_valid = CharDataset(names_valid, block_size=1, vocab=vocab)

bigram_model = CountingModel(block_size=1, vocab_size=len(vocab))
bigram_model.fit(bigram_train)
bigram_model.evaluate(bigram_valid)

The learned conditional probabilities can be visualized for the bigram model:

In [ ]:
plt.imshow(bigram_model.P);    # yellow pixel is for “qu” ☺

Similarly, we can fit a trigram model as follows:

In [ ]:
trigram_train = CharDataset(names_train, block_size=2, vocab=vocab)
trigram_valid = CharDataset(names_valid, block_size=2, vocab=vocab)
trigram_model = CountingModel(block_size=2, vocab_size=len(vocab))
trigram_model.fit(trigram_train)

# Sample prediction and evals
p = trigram_model(torch.tensor([[1, 1], [2, 2]]))
print(p.shape, p.sum(dim=1).view(-1))
print(trigram_model.evaluate(trigram_train))
print(trigram_model.evaluate(trigram_valid))

Both models are better than random:

In [ ]:
import math
math.log(len(vocab) - 2)    # -2 for <unk> and padding

Note that already at $n = 3$ generalization starts to drop:

In [ ]:
ngram_train = CharDataset(names_train, block_size=3, vocab=vocab)
ngram_valid = CharDataset(names_valid, block_size=3, vocab=vocab)
ngram_model = CountingModel(block_size=3, vocab_size=len(vocab))
ngram_model.fit(ngram_train)

# Sample prediction and evals
print(ngram_model.evaluate(ngram_train))
print(ngram_model.evaluate(ngram_valid))

<br>

## Generating names

For concreteness, we will consider a trigram model. First, a character is sampled from a [multinomial distribution](https://online.stat.psu.edu/stat504/book/export/html/667)[^multinomial] given the start context `..`. The context is appended with each sampled character, say `e`. Since our trigram model only uses the last 2 characters, the relevant context becomes `.e`. This is repeated until another `.` is sampled, signaling the end of a name.

[^multinomial]: Independent samples with $|\mathcal{V}|$ mutually exclusive outcomes, one for each character.

In [ ]:
def generate_name(
    model, 
    dataset: CharDataset, 
    min_len=2,
    max_len=100, 
    g=None, seed=2718
):
    """Generate names from a Markov process with cond prob from model."""
    if g is None:
        g = torch.Generator().manual_seed(seed)
    
    context = PAD_TOKEN * dataset.block_size
    out = []
    while len(context) < max_len:
        x = dataset.tokenizer.encode(context).view(1, -1)
        p = model(x)[0]
        j = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        c = dataset.tokenizer.decode(j)
        if c == PAD_TOKEN:
            if len(out) >= min_len:
                break
            else:
                continue
        
        out.append(c)
        context = context[1:] + c
    
    return "".join(out)

Sampling based on bigram counts:

In [ ]:
g = torch.Generator().manual_seed(0)
for i in range(10):
    print(generate_name(bigram_model, bigram_train, min_len=3, max_len=60, g=g))

Sampling based on trigram counts seem to have better results:

In [ ]:
g = torch.Generator().manual_seed(0)
for i in range(10):
    print(generate_name(trigram_model, trigram_train, min_len=3, max_len=60, g=g))

# Character embeddings

Recall that the *n*-gram language model, that derives the next token probability distribution based on *n*-gram counts in the training data, has an inherent limitation in the context size of *n* - 1 and with *n*-grams becoming sparse as *n* increases. 

In this section, we implement a language model that learns **vector embeddings** for each character (see [@Bengio2003]). This approach allows having contexts of arbitrarily length (e.g., concatenating the embeddings). Furthermore, generalization is obtained because a sequence of characters that has never been seen before gets an informative probability distribution if it is made of characters that are similar (in the sense of having a nearby representation) to those that formed a context in the training set. 

The network architecture is shown in the following figure:


![Neural network language model with embedding layer and block size of 3. An input string `"ner"` is passed to the embedding layer. Note that the embeddings are concatenated in the correct order. The resulting concatenation of embeddings are passed to the two-layer MLP with logits.](img/mlp-char-level.drawio.svg){#fig-mlp-char-level.drawio fig-align="center" width=600px}

The first component of the network is an **embedding matrix**, where each row corresponds to an embedding vector. Then, the embedding vectors are concatenated in the correct order and passed to the two-layer MLP to get the logits after a dense operation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

class MLP(nn.Module):
    def __init__(self, 
                 emb_size: int, 
                 width: int, 
                 block_size: int, 
                 vocab_size: int):
        
        super().__init__()
        self.C = nn.Embedding(vocab_size, emb_size)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(block_size * emb_size, width)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(width, vocab_size)

    def forward(self, x):
        x = self.flatten(self.C(x))
        h = self.relu(self.fc1(x))
        z = self.fc2(h)
        return z


model = MLP(emb_size=10, width=64, block_size=3, vocab_size=29)
summary(model, input_data=torch.tensor([[0, 0, 0] for _ in range(32)]))

**Remark.** It is recommended that weight decay, such as L2, is applied only to the weights and not on biases, embeddings, etc., so that network expressivity is not degraded. See [this post](https://discuss.pytorch.org/t/weight-decay-only-for-weights-of-nn-linear-and-nn-conv/114348) on how to implement this.

<br>

**Backpropagation.** Calculating the gradients of the embedding matrix:

In [ ]:
BLOCK_SIZE = 3
VOCAB_SIZE = 28
BATCH_SIZE = 4
x = torch.randint(low=0, high=VOCAB_SIZE, size=(BATCH_SIZE, BLOCK_SIZE))
y = torch.randint(low=0, high=VOCAB_SIZE, size=(BATCH_SIZE,))

C = torch.randn(VOCAB_SIZE, 10,     requires_grad=True)
W = torch.randn(BLOCK_SIZE * 10, 4, requires_grad=True)
b = torch.randn(4,                  requires_grad=True)

x̄     = C[x]       
x̄_cat = x̄.view(x̄.shape[0], -1)
y     = x̄_cat @ W + b

Observe that the gradient of $\bar{\boldsymbol{{\mathsf{x}}}}$ is just the gradient of $\bar{\boldsymbol{{\mathsf{x}}}}_{\text{cat}}$ reshaped. The embedding operation can be written as $\bar{\boldsymbol{{\mathsf{x}}}}_{btj} = \mathsf{C}_{\mathbf{x}_{bt}j}.$ Hence: 

$$
\begin{aligned}
\frac{\partial\mathcal{L}}{\partial\mathsf{C}_{ij}} 
&= \sum_b \sum_t \sum_k \frac{\partial\mathcal{L}}{\partial\bar{\boldsymbol{{\mathsf{x}}}}_{btk}} \frac{\partial\bar{\boldsymbol{{\mathsf{x}}}}_{btk}}{\partial\mathsf{C}_{ij}} \\
&= \sum_b \sum_t \sum_k \frac{\partial\mathcal{L}}{\partial\bar{\boldsymbol{{\mathsf{x}}}}_{btk}} \boldsymbol{\delta}_{\mathbf{x}_{bt}i} \boldsymbol{\delta}_{jk} \\
&= \sum_b \sum_t \boldsymbol{\delta}_{i\mathbf{x}_{bt}}\frac{\partial\mathcal{L}}{\partial\bar{\boldsymbol{{\mathsf{x}}}}_{btj}}.
\end{aligned}
$$

The last formula is simply an instruction on where to add the gradients of $\bar{\boldsymbol{{\mathsf{x}}}}$ for entries that are pulled out of $\mathsf{C}.$ However, note that we also sum over the batch index since these instances share the same embedding weights[^causal-structure].

[^causal-structure]: The formula exhibits no explicit causal structure based on the token order $t.$ But happens implicitly with the way the dataset and the network are structured, i.e. token order is preserved in each context window.

In [ ]:
for u in [x̄, x̄_cat, y]:
    u.retain_grad()

loss = (y ** 2).sum(dim=1).mean()
loss.backward()

It follows that the gradient of the embedding matrix is:

In [ ]:
δ = F.one_hot(x, num_classes=VOCAB_SIZE).view(-1, VOCAB_SIZE).float()
dC = δ.T @ x̄.grad.view(-1, 10)  # δ.T so (bt, i) -> (i, bt)

# Checking with autograd
print("maxdiff:", (dC - C.grad).abs().max().item(), "\texact:", torch.all(dC == C.grad).item())

In [ ]:
# Three input sequences in a batch (B = 4)
δ = F.one_hot(x, num_classes=VOCAB_SIZE).view(-1, VOCAB_SIZE).float().T
δ[:, 3: 6] += 1 # add shade
δ[:, 6: 9] += 2
δ[:, 9:12] += 3

fig, ax = plt.subplots(1, 2)
ax[0].imshow(δ.tolist(), interpolation="nearest", cmap="gray")
ax[0].set_xlabel("$(b, t)$")
ax[0].set_ylabel("$i$ (characters)")

ax[1].imshow(x̄.grad.view(-1, 10), interpolation="nearest")
ax[1].set_ylabel("$(b, t)$")
ax[1].set_xlabel("$j$ (embedding)");

**Figure.** Visualizing [Kronecker delta](https://en.wikipedia.org/wiki/Kronecker_delta) tensor (**left**) and gradient tensor (**right**) from the above equation. The tensor $\boldsymbol{\delta}_{i\mathbf{x}_{bt}}$ indicates all indices $(b, t)$ where the *i*-th character occurs.
All embedding vector gradients $\frac{\partial{\mathcal{L}}}{\partial\bar{\boldsymbol{{\mathsf{x}}}}_{bt:}}$ for the $i$th character are collapsed over all time steps to get 
$\frac{\partial{\mathcal{L}}}{\partial \mathsf{C}_{i:}}.$

<br>

## Model training

Reusing our [trainer engine](./06-cnn.html#trainer-engine) code:

In [ ]:
import numpy as np
from tqdm.notebook import tqdm
from contextlib import contextmanager
from torch.utils.data import DataLoader

DEVICE = "mps"


@contextmanager
def eval_context(model):
    """Temporarily set to eval mode inside context."""
    is_train = model.training
    model.eval()
    try:
        yield
    finally:
        model.train(is_train)


class Trainer:
    def __init__(self,
        model, optim, loss_fn, scheduler=None, callbacks=[],
        device=DEVICE, verbose=True
    ):
        self.model = model.to(device)
        self.optim = optim
        self.device = device
        self.loss_fn = loss_fn
        self.train_log = {"loss": [], "loss_avg": []}
        self.valid_log = {"loss": []}
        self.verbose = verbose
        self.scheduler = scheduler
        self.callbacks = callbacks
    
    def __call__(self, x):
        return self.model(x.to(self.device))

    def forward(self, batch):
        x, y = batch
        x = x.to(self.device)
        y = y.to(self.device)
        return self.model(x), y

    def train_step(self, batch):
        preds, y = self.forward(batch)
        loss = self.loss_fn(preds, y)
        loss.backward()
        self.optim.step()
        self.optim.zero_grad()
        return {"loss": loss}

    @torch.inference_mode()
    def valid_step(self, batch):
        preds, y = self.forward(batch)
        loss = self.loss_fn(preds, y, reduction="sum")
        return {"loss": loss}
    
    def run(self, epochs, train_loader, valid_loader):
        for e in tqdm(range(epochs)):
            for batch in train_loader:
                # optim and lr step
                output = self.train_step(batch)
                if self.scheduler:
                    self.scheduler.step()

                # step callbacks
                for callback in self.callbacks:
                    callback()

                # logs @ train step
                steps_per_epoch = len(train_loader)
                w = int(0.05 * steps_per_epoch)
                self.train_log["loss"].append(output["loss"].item())
                self.train_log["loss_avg"].append(np.mean(self.train_log["loss"][-w:]))

            # logs @ epoch
            output = self.evaluate(valid_loader)
            self.valid_log["loss"].append(output["loss"])
            if self.verbose:
                print(f"[Epoch: {e+1:>0{int(len(str(epochs)))}d}/{epochs}]    loss: {self.train_log['loss_avg'][-1]:.4f}    val_loss: {self.valid_log['loss'][-1]:.4f}")

    def evaluate(self, data_loader):
        with eval_context(self.model):
            valid_loss = 0.0
            for batch in data_loader:
                output = self.valid_step(batch)
                valid_loss += output["loss"].item()

        return {"loss": valid_loss / len(data_loader.dataset)}

    @torch.inference_mode()
    def predict(self, x: torch.Tensor):
        with eval_context(self.model):
            return self(x)

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

DEVICE = "mps"
PAD_TOKEN = "."
BATCH_SIZE = 128
names = load_surnames()
vocab = Vocab(text="".join(names), preprocess=False, reserved_tokens=[PAD_TOKEN])
split_point = int(0.80 * len(names))

train_dataset = CharDataset(names[:split_point], block_size=BLOCK_SIZE, vocab=vocab)
valid_dataset = CharDataset(names[split_point:], block_size=BLOCK_SIZE, vocab=vocab)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

epochs = 15
model = MLP(emb_size=3, width=64, block_size=BLOCK_SIZE, vocab_size=len(vocab)).to(DEVICE)
loss_fn = F.cross_entropy
optim = torch.optim.AdamW(model.parameters(), lr=0.001)
scheduler = OneCycleLR(optim, max_lr=0.01, steps_per_epoch=len(train_loader), epochs=epochs)
trainer = Trainer(model, optim, loss_fn, scheduler, device=DEVICE)
trainer.run(epochs=epochs, train_loader=train_loader, valid_loader=valid_loader)

This performs better than the 4-gram count model.

In [ ]:
from matplotlib.ticker import StrMethodFormatter

num_epochs = len(trainer.valid_log["loss"])
num_steps_per_epoch = len(trainer.train_log["loss"]) // num_epochs

plt.figure(figsize=(6, 4))
plt.gca().yaxis.set_major_formatter(StrMethodFormatter("{x:,.2f}")) # 2 decimal places
plt.plot(trainer.train_log["loss"], alpha=0.6, color="C0")
plt.plot(trainer.train_log["loss_avg"], color="C0", label="train")
plt.plot(list(range(num_steps_per_epoch, (num_epochs + 1) * num_steps_per_epoch, num_steps_per_epoch)), trainer.valid_log["loss"], label="valid", color="C1")
plt.ylabel("loss")
plt.xlabel("steps")
plt.grid(linestyle="dotted")
plt.legend();

Generated names are starting to look natural:

In [ ]:
for _ in range(10):
    n = generate_name(lambda x: F.softmax(model.to("cpu")(x), dim=1), train_dataset, seed=_)
    print(n)

Looking at the trained character embeddings:

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 4))
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1])
ax1 = fig.add_subplot(gs[0], projection="3d")
ax2 = fig.add_subplot(gs[1])
chars = list(train_dataset.vocab.tokens)

# Scatter plot the embeddings; annotate
embeddings = model.C.weight.data.detach().cpu().numpy()
ax1.scatter(embeddings[:, 0], embeddings[:, 1], embeddings[:, 2])
ax1.set_title("Trained embeddings")
for i, c in enumerate(chars):
    c = "?" if c == "<unk>" else c
    ax1.text(embeddings[i, 0], embeddings[i, 1], embeddings[i, 2], c)

# Apply t-SNE
PP = 3
tsne = TSNE(n_components=2, perplexity=PP, random_state=42)
tsne_emb = tsne.fit_transform(model.C.weight.data)

ax2.scatter(tsne_emb[:, 0], tsne_emb[:, 1], c="green", alpha=0.5)
ax2.axis("equal")
ax2.set_title(f"t-SNE (n=2, pp={PP})")
for i, c in enumerate(chars):
    c = "?" if c == "<unk>" else c
    ax2.text(tsne_emb[i, 0] + 1.0, tsne_emb[i, 1] + 1.0, c)

fig.tight_layout()

**Figure.** Vowels are isolated, as well as `_` and `.`. Interestingly, the unknown token ▮ is located near other letters. In general, tight clustering of characters in t-SNE may indicate that these characters are not fully understood by the network[^tsne-clustering].

[^tsne-clustering]: A tight clustering of embedding vectors hints that embedding dimension may be a bottleneck, and therefore can be increased to improve performance. Indeed, performance improved after increasing embedding dimension from 2 (not shown) to 3 with the rest of the variables fixed.

# Temporal / Causal convolutions

One problem with our previous network is that sequential information from the inputs are mixed or squashed too fast (i.e. in one layer). We can make this network deeper by adding dense layers, but it still does not solve this problem. In this section, we implement a convolutional neural network architecture similar to **WaveNet** [@wavenet]. This allows the character embeddings to be fused slowly.

![[@wavenet] Tree-like structure formed by a stack of dilated *causal* convolutional layers. The term causal is used since the network is constrained so that an output node at position `t` can only depend on input nodes at position `t-k:t`.](img/04-wavenet.png){#fig-wavenet width=700px}

The structure of the network allows us to use a larger block size. 
Previously, increasing block size by 1 means that the network 
width increases 
equal to the embedding dimension.
For this network, the width is fixed but we have to increase depth.

In [ ]:
PAD_TOKEN = "."
BLOCK_SIZE = 8  # !
names = load_surnames()
vocab = Vocab(text="".join(names), preprocess=False, reserved_tokens=[PAD_TOKEN])
split_point = int(0.80 * len(names))

train_dataset = CharDataset(names[:split_point], block_size=BLOCK_SIZE, vocab=vocab)
valid_dataset = CharDataset(names[split_point:], block_size=BLOCK_SIZE, vocab=vocab)

for x, y in zip(train_dataset.xs, train_dataset.ys):
    print("".join(x), "-->", y)
    if y == PAD_TOKEN: 
        break

To implement this without using dilated convolutions, we define the following class 
so that only two characters are combined at each step. The linear layer is applied to 
the last dimension which the following layer expands. Here `n` corresponds to the number 
of characters that are combined at each step.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FlattenConsecutive(nn.Module):
    def __init__(self, n: int):
        super().__init__()
        self.n = n
    
    def forward(self, x):
        B, T, C = x.shape   # (batch, char, emb)
        x = x.view(B, T // self.n, C * self.n)
        return x

To see how this works:

In [ ]:
B, T, C = 2, 4, 2
x = torch.rand(B, T, C)
x

In [ ]:
x.view(B, T // 2, C * 2)

Information from characters can flow better through the network due to its hierarchical nature. 
Hence, we increase embedding size, network width, as well as make the network is deeper. 
Note that we need three layers to combine all characters, i.e. 2 × 2 × 2 = 8 (block size). Stride happens in the way character blocks are fed to the layers, so convolutions need not be explicitly used. The resulting network resembles our previous network:

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

emb_size = 24
width = 128
DEVICE = "mps"
VOCAB_SIZE = len(vocab)
BATCH_SIZE = 128

wavenet = nn.Sequential(
    nn.Embedding(VOCAB_SIZE, emb_size),
    FlattenConsecutive(2), nn.Linear(emb_size * 2, width), nn.ReLU(),
    FlattenConsecutive(2), nn.Linear(   width * 2, width), nn.ReLU(),
    FlattenConsecutive(2), nn.Linear(   width * 2, width), nn.ReLU(),
    nn.Linear(width, VOCAB_SIZE), nn.Flatten()  # flatten: rank 3 -> rank 2
)

epochs = 5
wavenet = wavenet.to(DEVICE)
loss_fn = F.cross_entropy
optim = torch.optim.AdamW(wavenet.parameters(), lr=0.0001)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
scheduler = OneCycleLR(optim, max_lr=0.01, steps_per_epoch=len(train_loader), epochs=epochs)
trainer = Trainer(wavenet, optim, loss_fn, scheduler, device=DEVICE)
trainer.run(epochs=epochs, train_loader=train_loader, valid_loader=valid_loader)

**Remark.** Finally broke through 2.3 loss!

In [ ]:
from matplotlib.ticker import StrMethodFormatter

num_epochs = len(trainer.valid_log["loss"])
num_steps_per_epoch = len(trainer.train_log["loss"]) // num_epochs

a = np.array(trainer.train_log["loss_avg"])[0] + 0.1
b = np.array(trainer.train_log["loss_avg"])[-1] - 0.1
plt.figure(figsize=(6, 4))
plt.gca().yaxis.set_major_formatter(StrMethodFormatter("{x:,.2f}")) # 2 decimal places
plt.plot(np.array(trainer.train_log["loss"]), alpha=0.6, color="C0")
plt.ylabel("loss")
plt.xlabel("steps")
plt.plot(np.array(trainer.train_log["loss_avg"]), color="C0", label="train")
plt.plot(list(range(num_steps_per_epoch, (num_epochs + 1) * num_steps_per_epoch, num_steps_per_epoch)), trainer.valid_log["loss"], label="valid", color="C1")
plt.grid(linestyle="dotted")
plt.legend();

The generated names look very natural:

In [ ]:
for _ in range(10):
    n = generate_name(lambda x: F.softmax(wavenet.to("cpu")(x), dim=1), train_dataset, seed=_)
    print(n)

# Appendix: Neural *n*-gram model

A single layer FNN with weight $\mathbf{W}$ of shape $|\mathcal{V}|^n$ can also be used to model *n*-grams. This weight tensor corresponds to the count matrix corresponding to the frequency of each *n*-gram.
The relevant row of weights is picked out by computing $\mathbf{w}_a = \mathbf{x}_a \mathbf{W}$ where $\mathbf{x}_a$ is the one-hot encoding of the input character $a.$ Then, we apply softmax to $\mathbf{w}_a$ which is a common technique for converting real-valued vectors to probabilities.


For concreteness, we implement a neural bigram model:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BigramNet(nn.Module):
    def __init__(self, vocab_size, alpha=0.1, seed=2147483647):
        super().__init__()
        self.vocab_size = vocab_size
        self.g = torch.Generator().manual_seed(seed)
        self.W = nn.Parameter(torch.randn((vocab_size, vocab_size), generator=self.g, requires_grad=True))
        self.alpha = alpha

    def forward(self, x):
        """Returns p(· | x) over characters."""
        xenc = F.one_hot(x, num_classes=self.vocab_size).float()
        logits = xenc.view(-1, self.vocab_size) @ self.W
        counts = logits.exp()
        probs = (counts + self.alpha) / (counts + self.alpha).sum(1, keepdim=True)
        return probs

The weights with values in `(-inf, +inf)` can be interpreted as **log-counts** of bigrams that start on the row index. This is interesting because growth in the negative direction is different from growth in the positive direction. Applying `.exp()` we get units of count with values in `(0, +inf)` which we interpret as counts. Normalizing these results in a probability vector.


![Schematic diagram of the bigram neural net. Computing the probability assigned by the model on `b` given `a` which is represented here as one-hot vector. Note that this is also the backward dependence for a bigram `ab` encountered during training with the NLL loss.](img/bigram-nn.drawio.svg){#fig-bigram-nn.drawio fig-align="center" width=450px}

<br>

## Model training

The model is initialized with uniform conditional probabilities. Then, it is trained to minimize next character NLL (i.e. maximize the likelihood of train bigrams). We expect similar performance with the *n*-gram count model which has the same capacity.

In [ ]:
from tqdm.notebook import tqdm
from torch.optim import SGD
from torch.utils.data import DataLoader

names = load_surnames()
PAD_TOKEN = "."
vocab = Vocab(text="".join(names), preprocess=False, reserved_tokens=[PAD_TOKEN])

split_point = int(0.80 * len(names))
bigram_train = CharDataset(names[:split_point], block_size=1, vocab=vocab)
bigram_valid = CharDataset(names[split_point:], block_size=1, vocab=vocab)
train_loader = DataLoader(bigram_train, batch_size=256, shuffle=True)
valid_loader = DataLoader(bigram_valid, batch_size=256, drop_last=True)

model = BigramNet(vocab_size=len(vocab))
optim = SGD(model.parameters(), lr=10.0)

losses = []
num_steps = 500
for k in tqdm(range(num_steps)):
    x, y = next(iter(train_loader))
    p = model(x)
    loss = -p[torch.arange(len(y)), y].log().mean()  # next char nll

    loss.backward()
    optim.step()
    model.zero_grad()
    
    # logging
    losses.append(loss.item())

In [ ]:
print(f"{sum(losses[-10:]) / 10: .4f}", "(avg loss @ last 10 steps)")
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.ylabel('loss')
plt.xlabel('step');

We get the expected validation performance:

In [ ]:
loss = 0.0
for x, y in valid_loader:
    p = model(x)
    loss += -p[torch.arange(len(y)), y].log().sum()

B = x.shape[0]
print(loss / (len(valid_loader) * B))

<br>

## Sampling

Generating names and its associated NLL:

In [ ]:
s = []
for x in torch.tensor([[0, 1, 2], [3, 4, 5]]).tolist():
    s.append(bigram_train.tokenizer.decode(x))
s

In [ ]:
def name_loss(name, model, dataset):
    nll = 0.0
    for c in name:
        x = torch.tensor(dataset.tokenizer.encode(c)).long().view(-1, 1)
        p = model(x)[0, dataset.vocab[c]]
        nll += -math.log(p)
    return nll / (len(name) + 1)

sample = list(filter(lambda n: len(n) >= 2, [generate_name(model, bigram_train, seed=_) for _ in range(12)]))
name_losses = {n: name_loss(n, model, bigram_train) for n in sample}
for n in sorted(sample, key=lambda n: name_losses[n]):
    print(f"{n:<50} {name_losses[n]:.3f}")

<br>

Recall that instead of maximizing names, we are maximizing next character likelihood. Hence, we can get names with low NLL (relative to the random baseline) even if the name is unlikely to occur naturally[^generative-strength]. Consequently, it should be rare that a generated name is in the training dataset:

[^generative-strength]: Depending on the context, this can be considered as a strength of generative models.

In [ ]:
print(r"Generated names found in train dataset:")
n = 1000
sample = [generate_name(model, bigram_train, seed=i) for i in range(n)]
print(f"{100 * sum([n in names for n in sample]) / n}% (n={n})")

<br>

From the conditional distributions, it looks like the neural net recovered the count matrix!

In [ ]:
bigram_model = CountingModel(block_size=1, vocab_size=len(vocab))
bigram_model.fit(bigram_train)

counts = model.W.exp()
P = (counts / counts.sum(dim=1, keepdim=True)).data
P2 = bigram_model.P
fig, ax = plt.subplots(1, 2)
fig.tight_layout()
ax[0].imshow(P2)
ax[0].set_title("count")
ax[1].imshow(P)
ax[1].set_title("NN");

# Appendix: Autoregressive models

The goal of [autoregressive models](https://en.wikipedia.org/wiki/Autoregressive_model) is to characterize 
$p(\mathbf{x}_t \mid \mathbf{x}_1, \ldots, \mathbf{x}_{t-1})$, i.e. the next element is estimated using previous elements of the same sequence. Note that the 
entire distribution is generally hard to compute, and we may be content with $\mathbb{E}\left[\mathbf{x}_t \mid \mathbf{x}_1, \ldots, \mathbf{x}_{t-1}\right]$, i.e. estimating the average value of the next element.
One issue is that the length of sequences increase with the amount of data that we encounter.
Much of sequence modeling literature revolve around techniques for dealing with 
increasing context size to predict the next token or certain statistics of the distribution $p(\mathbf{x}_t \mid \mathbf{x}_1, \ldots, \mathbf{x}_{t-1}).$

A natural strategy is to just ignore more data, i.e. only use past $\tau$ observations, so that we estimate $p(\mathbf{x}_t \mid \mathbf{x}_{t-\tau}, \ldots, \mathbf{x}_{t-1}).$ This is a [Markovian assumption](https://en.wikipedia.org/wiki/Markov_model) where we assume that the past $\tau$ elements are sufficient to approximate the next element. This makes sense especially for phenomenon where long-range dependency is rare, or that the importance of long-range dependency decays quickly with time.
In this case, all inputs are of length $\tau$
which allows us to train any linear model or deep network that requires fixed-length vectors as inputs.

## Autoregressive linear model

Let's train an autogressive *linear* model with 4th order Markov condition.

In [ ]:
import torch
torch.manual_seed(42)

B = 16
T = 1000
N = 600
tau = 4
t = torch.arange(1, T + 1, dtype=torch.float32)
x = torch.sin(0.01 * t) + torch.randn(T) * 0.2

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "svg"
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(t, x)
plt.ylabel("$x$")
plt.xlabel("$t$");

Our task is to model $[\mathbf{x}_{t - \tau}, \ldots, \mathbf{x}_{t - 1}] \mapsto \mathbf{x}_{t}.$ Here we sample windows of fixed length $\tau = 4$:

In [ ]:
def build_dataset(tau: int):
    # stack offsets each with length T - tau
    f = torch.stack([x[i: T-tau+i] for i in range(tau)], 1)
    y = x[tau: ].reshape(-1, 1)
    return f, y


f, y = build_dataset(tau=4)
print(f.shape)
print(f)

We will discard $\tau$ elements to get the targets, which start with $t = \tau  + 1.$

In [ ]:
y[0]    # [0.3954, 0.3175,  0.2101, -0.3811,] 0.2197 <- first target

Creating the data loader:

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

data = TensorDataset(f[:-100], y[:-100])
inp, tgt = next(iter(data))
inp, tgt

Training a linear model which uses previous $\tau = 4$ points as input:

In [ ]:
import torch.nn as nn

def train_model(
    f: torch.tensor, 
    y: torch.tensor,
    tau: int
):
    """Train a tau-order linear Markov model."""
    data = TensorDataset(f, y)
    model = nn.Sequential(nn.Linear(tau, 1))
    optim = torch.optim.SGD(model.parameters(), lr=0.01)
    for _ in range(3):
        for u, v in DataLoader(data, batch_size=16, shuffle=True):
            loss = ((model(u) - v) ** 2).mean()
            loss.backward()
            optim.step()
            optim.zero_grad()
    
    return model


model = train_model(f[:-100], y[:-100], tau=4)

The model does not simple average:

In [ ]:
print(list(model.parameters())[0])

Let's compare with a baseline averaging model:

In [ ]:
y_baseline = f.mean(dim=1)

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(y, label="$y$")
plt.plot(model(f).reshape(-1).tolist(), label="4th-order")
plt.plot(y_baseline, label="baseline")
plt.axvline(len(f[:-100]), linestyle="dashed", color="k", label="[test split]")
plt.legend(fontsize=8);

Also train a first-order Markov model for good fun:

In [ ]:
# First order markov model
f1, y1 = build_dataset(tau=1)
model1 = train_model(f1[:-100], y1[:-100], tau=1)

Test performance:

In [ ]:
print("Mean squared error")
print(f"  (4th-order)  {((model(f[-100:])   - y [-100:]) ** 2).mean().item():.3f}")
print(f"  (1st-order)  {((model1(f1[-100:]) - y1[-100:]) ** 2).mean().item():.3f}")
print(f"  (base-line)  {((y_baseline[-100:] - y [-100:]) ** 2).mean().item():.3f}")

<br>

## Look-ahead predictions

Suppose we have measurement data up to $t = 204$ and we want to predict $t = 205, \ldots, 209$. To do this we can reuse predictions. Beginning precisely $t = 209$, we have to use solely predicted values instead of observed data:

$$
\begin{aligned}
& \hat{x}_{205}=f\left(x_{201}, x_{202}, x_{203}, x_{204}\right) \\
& \hat{x}_{206}=f\left(x_{202}, x_{203}, x_{204}, \hat{x}_{205}\right) \\
& \hat{x}_{207}=f\left(x_{203}, x_{204}, \hat{x}_{205}, \hat{x}_{206}\right) \\
& \hat{x}_{208}=f\left(x_{204}, \hat{x}_{205}, \hat{x}_{206}, \hat{x}_{207}\right) \\
& \hat{x}_{209}=f\left(\hat{x}_{205}, \hat{x}_{206}, \hat{x}_{207}, \hat{x}_{208}\right)
\end{aligned}
$$

This approach immediately accumulates errors and we get nonsense predictions:

In [ ]:
t0 = 201
preds = []
k_max = 800
ctx = x[t0: t0 + tau].tolist()
for i in range(k_max):
    x_hat = model(torch.tensor(ctx).reshape(1, -1))[0]
    preds.append(x_hat.item())
    ctx = ctx[1:] + [x_hat]

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(t[tau:], y, label="target")
plt.plot(range(t0, t0 + k_max), preds, label="multi-step preds", linestyle="dashed")
plt.grid(linestyle="dotted")
plt.legend(fontsize=8);

<br>

## Appendix: Project Gutenberg Reader

In this section, we generalize the `TimeMachine` class to read any title in Project Gutenberg.

In [ ]:
import re
import os
import requests

ENCODING = "utf-8-sig"


class ProjectGutenberg:
    def __init__(self, url: str, data_dir: str, download=False, token_level="char"):
        self.token_level = token_level
        self.filepath = f"{data_dir}/{url.split('/')[-1]}"
        if download or not os.path.exists(self.filepath):
            self.download(url, self.filepath)

    @staticmethod
    def get_title(filepath):
        with open(filepath, "r", encoding=ENCODING) as f:
            line = f.readline()
        prefix = "The Project Gutenberg eBook of"
        return line.replace(prefix, "").strip()

    @staticmethod
    def download(url, filepath):
        print(f"Downloading text from {url} ...", end=" ")
        response = requests.get(url, stream=True)
        response.raise_for_status()
        print("OK!")
        with open(filepath, "wb") as output:
            output.write(response.content)
        
    @staticmethod
    def preprocess(text: str, title: str):
        s = f"*** START OF THE PROJECT GUTENBERG EBOOK {title.upper()} ***"
        e = f"*** END OF THE PROJECT GUTENBERG EBOOK {title.upper()} ***"
        text = text[text.find(s) + len(s): text.find(e)]
        text = re.sub('[^A-Za-z]+', ' ', text).lower().strip()
        return text
    
    @staticmethod
    def tokenize(text: str, token_level="char"):
        return list(text) if token_level == "char" else text.split()

    def build(self, vocab=None):
        with open(self.filepath, "r", encoding=ENCODING) as f:
            raw_text = f.read()
        
        self.title = self.get_title(self.filepath)
        self.text = self.preprocess(raw_text, self.title)
        self.tokens = self.tokenize(self.text, self.token_level)

        vocab = Vocab(self.tokens) if vocab is None else vocab
        corpus = vocab[self.tokens]
        return corpus, vocab

The list of [most downloaded books](https://www.gutenberg.org/browse/scores/top) can be obtained by web scraping:

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_top100_books():
    url = "https://www.gutenberg.org/browse/scores/top#books-last30"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    # scraping Top 100 EBooks yesterday section
    header = soup.find(id="books-last1")
    booklist = header.find_next("ol").find_all("a", href=True)

    # build url for plain text files
    text_urls = []
    base_url = "https://www.gutenberg.org"
    for link in booklist:
        idx = link["href"].split("/")[-1]
        text_urls.append(f"{base_url}/cache/epub/{idx}/pg{idx}.txt")

    return text_urls

In [ ]:
urls = get_top100_books()
print(len(urls))
urls[:10]